# Zeeschuimer Data Import [![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.8199901.svg)](https://doi.org/10.5281/zenodo.8199901)

![Notes on (Computational) Social Media Research Banner](https://raw.githubusercontent.com/michaelachmann/social-media-lab/main/images/banner.png)

## Overview

This Jupyter notebook is a part of the social-media-lab.net project, which is a work-in-progress textbook on computational social media analysis. The notebook is intended for use in my classes.

The **Zeeschuimer Data Import** notebook handles *ndjson* files provided by the [Zeeschuimer](https://github.com/digitalmethodsinitiative/zeeschuimer) plugin for collecting Instagram posts. After importing the files, we can download images (same as 4CAT). Additionally we can convert the data to be compatible with other notebooks in the coures.

**TODO:** At the moment we can only download one image per posts. Future versions of this notebook should be capable of downloading all images / media from albums.

See [social-media-lab.net](https://social-media-lab.net/data-collection/ig-posts.html#zeeschuimer-4cat) for more information.

### Project Information

- Project Website: [social-media-lab.net](https://social-media-lab.net/)
- GitHub Repository: [https://github.com/michaelachmann/social-media-lab](https://github.com/michaelachmann/social-media-lab)

## License Information

This notebook, along with all other notebooks in the project, is licensed under the following terms:

- License: [GNU General Public License version 3.0 (GPL-3.0)](https://www.gnu.org/licenses/gpl-3.0.de.html)
- This Notebook incorporates code taken from the [4CAT repository](https://github.com/digitalmethodsinitiative/4cat/), licenced under Mozilla Public License, 2.0.  
- License File: [LICENSE.md](https://github.com/michaelachmann/social-media-lab/blob/main/LICENSE.md)


## Citation

If you use or reference this notebook in your work, please cite it appropriately. Here is an example of the citation:

```
Michael Achmann. (2023). michaelachmann/social-media-lab: 06.11.2023 (v0.0.3). Zenodo. https://doi.org/10.5281/zenodo.8199901
```

In [ ]:
import pandas as pd
import json

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from tqdm.auto import tqdm
import json
import pandas as pd
import datetime

import_format = "4CAT"
import_filename = '/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/clean_girl_zeeschuimer-export-pinterest.com-2026-01-26T142706.ndjson'
export_filename = '/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/2026-01-26_cleangirl.csv'


# ---------------------------------------------------------
# Pinterest Parser
# ---------------------------------------------------------

def extract_image_url(node):
    """Extract best available image URL from Pinterest pin."""
    images = node.get("images", {})

    if not isinstance(images, dict):
        return None

    # Preferred order
    for key in ["orig", "736x", "474x"]:
        if key in images and isinstance(images[key], dict):
            return images[key].get("url")

    return None


def parse_pinterest_item(node):
    """Convert a Pinterest pin into a clean, flat dictionary."""

    pin_id = node.get("id", node.get("node_id", ""))
    title = node.get("grid_title", "")
    description = (node.get("description", "") or "").strip()
    domain = node.get("domain", "")
    created_at = node.get("created_at", None)

    # Board name
    board_name = ""
    if isinstance(node.get("board"), dict):
        board_name = node["board"].get("name", "")

    # Image URL
    image_url = extract_image_url(node)

    return {
        "pin_id": pin_id,
        "title": title,
        "description": description,
        "domain": domain,
        "board": board_name,
        "created_at": created_at,
        "image_url": image_url,
    }


def map_item(item):
    """Route items to the correct parser."""
    if item.get("type") == "pin":
        return parse_pinterest_item(item)
    return None


# ---------------------------------------------------------
# Load NDJSON
# ---------------------------------------------------------

data_list = []

with open(import_filename, 'r') as file:
    for line in file:
        json_line = json.loads(line)
        data_field = json_line.get('data', json_line)
        data_list.append(data_field)

print("Total items loaded:", len(data_list))


# ---------------------------------------------------------
# Convert to table
# ---------------------------------------------------------

data = []

for element in tqdm(data_list):
    item = map_item(element)
    if item:
        data.append(item)

posts_cleangirl_df = pd.DataFrame(data)
posts_cleangirl_df.to_csv(export_filename, index=False)

print(f"Saved Pinterest CSV to {export_filename}")
print("Rows in CSV:", len(posts_cleangirl_df))

Total items loaded: 416


  0%|          | 0/416 [00:00<?, ?it/s]

Saved Pinterest CSV to /content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/2026-01-26_cleangirl.csv
Rows in CSV: 416


In [ ]:
posts_cleangirl_df.head()

,pin_id,title,description,domain,board,created_at,image_url
0,21251429487397507,,,Uploaded by user,Lifestyle,"Wed, 22 Oct 2025 07:20:25 +0000",https://i.pinimg.com/originals/09/e9/a3/09e9a3...
1,281543725716857,,,Uploaded by user,aesthetic inspo,"Sun, 22 Jun 2025 21:00:24 +0000",https://i.pinimg.com/originals/e2/99/49/e29949...
2,Ad4Jj0M0b2zAQG3867WSlOAYg40oMTT84o3eKc5sz0NwHb...,syoss Intense Keratin,Entdecke jetzt die syoss Intense Keratin Pfleg...,syoss.de,Ad-only Pins,"Thu, 08 Jan 2026 13:07:16 +0000",https://i.pinimg.com/originals/49/2c/90/492c90...
3,AU2ScxmiEIV1SpLi8wNppJ4U0vFgnZM3dtKi96S7qDYJE4...,Dein neuer Long-Hair Fav!✨,"Langes Haar? Hol dir die Pflege, die es verdie...",dm.de,Nur als Anzeigen gedachte Pins,"Thu, 20 Nov 2025 17:54:23 +0000",https://i.pinimg.com/originals/30/43/82/304382...
4,119556565105668720,,,Uploaded by user,2026 vision board,"Fri, 02 Jan 2026 05:46:44 +0000",https://i.pinimg.com/originals/e5/fb/d9/e5fbd9...


In [ ]:
len(posts_cleangirl_df)
posts_cleangirl_df = posts_cleangirl_df.iloc[:-116]
posts_cleangirl_df.to_csv(export_filename, index=False)

In [ ]:
from tqdm.auto import tqdm
import json
import pandas as pd
import datetime

import_format = "4CAT"
import_filename = '/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/messy_girl_zeeschuimer-export-pinterest.com-2026-01-26T142628.ndjson'
export_filename = '/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/2026-01-26_messygirl.csv'


# ---------------------------------------------------------
# Pinterest Parser
# ---------------------------------------------------------

def extract_image_url(node):
    """Extract best available image URL from Pinterest pin."""
    images = node.get("images", {})

    if not isinstance(images, dict):
        return None

    # Preferred order
    for key in ["orig", "736x", "474x"]:
        if key in images and isinstance(images[key], dict):
            return images[key].get("url")

    return None


def parse_pinterest_item(node):
    """Convert a Pinterest pin into a clean, flat dictionary."""

    pin_id = node.get("id", node.get("node_id", ""))
    title = node.get("grid_title", "")
    description = (node.get("description", "") or "").strip()
    domain = node.get("domain", "")
    created_at = node.get("created_at", None)

    # Board name
    board_name = ""
    if isinstance(node.get("board"), dict):
        board_name = node["board"].get("name", "")

    # Image URL
    image_url = extract_image_url(node)

    return {
        "pin_id": pin_id,
        "title": title,
        "description": description,
        "domain": domain,
        "board": board_name,
        "created_at": created_at,
        "image_url": image_url,
    }


def map_item(item):
    """Route items to the correct parser."""
    if item.get("type") == "pin":
        return parse_pinterest_item(item)
    return None


# ---------------------------------------------------------
# Load NDJSON
# ---------------------------------------------------------

data_list = []

with open(import_filename, 'r') as file:
    for line in file:
        json_line = json.loads(line)
        data_field = json_line.get('data', json_line)
        data_list.append(data_field)

print("Total items loaded:", len(data_list))


# ---------------------------------------------------------
# Convert to table
# ---------------------------------------------------------

data = []

for element in tqdm(data_list):
    item = map_item(element)
    if item:
        data.append(item)

posts_messygirl_df = pd.DataFrame(data)
posts_messygirl_df.to_csv(export_filename, index=False)

print(f"Saved Pinterest CSV to {export_filename}")
print("Rows in CSV:", len(posts_messygirl_df))

Total items loaded: 315


  0%|          | 0/315 [00:00<?, ?it/s]

Saved Pinterest CSV to /content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/2026-01-26_messygirl.csv
Rows in CSV: 315


In [ ]:
posts_messygirl_df.head()

,pin_id,title,description,domain,board,created_at,image_url
0,4362930883988158,,All posts • Instagram,instagram.com,2025 vision board,"Wed, 23 Jul 2025 11:31:46 +0000",https://i.pinimg.com/originals/48/df/6a/48df6a...
1,51087777020120409,messy girl,,Uploaded by user,messy girl,"Sat, 08 Nov 2025 09:03:42 +0000",https://i.pinimg.com/originals/ca/b0/64/cab064...
2,865817097129438495,brat core,#messygirl #aesthetic #inspiration #visionboard,Uploaded by user,the things that represent me the most,"Sun, 18 Jan 2026 16:11:02 +0000",https://i.pinimg.com/originals/1e/16/13/1e1613...
3,2392606048282786,,,Uploaded by user,VISION BROAD ig,"Fri, 22 Aug 2025 04:19:21 +0000",https://i.pinimg.com/originals/f0/50/79/f05079...
4,36521446972877706,,chessa subbiondo,Uploaded by user,full body skinny,"Mon, 24 Nov 2025 17:05:04 +0000",https://i.pinimg.com/originals/b3/05/db/b305db...


In [ ]:
len(posts_messygirl_df)
posts_messygirl_df = posts_messygirl_df.iloc[:-15]
posts_messygirl_df.to_csv(export_filename, index=False)

In [ ]:
import pandas as pd

# CSVs einlesen
df_clean = pd.read_csv("/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/2026-01-26_cleangirl.csv")
df_messy = pd.read_csv("/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/2026-01-26_messygirl.csv")

# Labels hinzufügen
df_clean["category"] = "clean_girl_"
df_messy["category"] = "messy_girl_"

# Untereinander zusammenfügen
df_combined = pd.concat([df_clean, df_messy], ignore_index=True)

# Optional: speichern
df_combined.to_csv("/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/combined_clean_messy_new.csv", index=False)

df_combined.head()

,pin_id,title,description,domain,board,created_at,image_url,category
0,21251429487397507,NaN,NaN,Uploaded by user,Lifestyle,"Wed, 22 Oct 2025 07:20:25 +0000",https://i.pinimg.com/originals/09/e9/a3/09e9a3...,clean_girl_
1,281543725716857,NaN,NaN,Uploaded by user,aesthetic inspo,"Sun, 22 Jun 2025 21:00:24 +0000",https://i.pinimg.com/originals/e2/99/49/e29949...,clean_girl_
2,Ad4Jj0M0b2zAQG3867WSlOAYg40oMTT84o3eKc5sz0NwHb...,syoss Intense Keratin,Entdecke jetzt die syoss Intense Keratin Pfleg...,syoss.de,Ad-only Pins,"Thu, 08 Jan 2026 13:07:16 +0000",https://i.pinimg.com/originals/49/2c/90/492c90...,clean_girl_
3,AU2ScxmiEIV1SpLi8wNppJ4U0vFgnZM3dtKi96S7qDYJE4...,Dein neuer Long-Hair Fav!✨,"Langes Haar? Hol dir die Pflege, die es verdie...",dm.de,Nur als Anzeigen gedachte Pins,"Thu, 20 Nov 2025 17:54:23 +0000",https://i.pinimg.com/originals/30/43/82/304382...,clean_girl_
4,119556565105668720,NaN,NaN,Uploaded by user,2026 vision board,"Fri, 02 Jan 2026 05:46:44 +0000",https://i.pinimg.com/originals/e5/fb/d9/e5fbd9...,clean_girl_


In [ ]:
len(df_combined)

600

In [ ]:
import requests
from pathlib import Path
import pandas as pd

image_folder = "/content/drive/MyDrive/Team Dattel/Computational Social Media Analysis/Preprocessing Data/Images"

# Set up the base path for images
base_image_path = Path(image_folder)
base_image_path.mkdir(parents=True, exist_ok=True)

with requests.Session() as session:
    for index, row in df_combined.iterrows():

        media_url = row["image_url"]
        pin_id = row["pin_id"]
        category = row["category"].replace(" ", "_")  # Leerzeichen vermeiden

        # Skip rows without image URL
        if pd.isna(media_url) or not isinstance(media_url, str):
            print(f"Skipping row {index}: no image URL")
            continue

        # Filename: category_pinid.jpg
        filename = f"{category}_{pin_id}.jpg"
        file_path = base_image_path / filename

        try:
            response = session.get(media_url, allow_redirects=True)
            response.raise_for_status()

            with open(file_path, "wb") as f:
                f.write(response.content)

        except requests.HTTPError as e:
            print(f"HTTP Error for {media_url}: {e}")

        except requests.RequestException as e:
            print(f"Request Exception for {media_url}: {e}")

## ZIP files for download

In [ ]:
!zip -r posts.zip /content/posts/

  adding: content/posts/ (stored 0%)
  adding: content/posts/images/ (stored 0%)
  adding: content/posts/images/jungealternativebayern/ (stored 0%)
  adding: content/posts/images/jungealternativebayern/CzGC5dxth82.jpg (deflated 1%)
  adding: content/posts/images/kathaschulze/ (stored 0%)
  adding: content/posts/images/kathaschulze/CzEIzgitG5w.jpg (deflated 1%)
  adding: content/posts/images/spdde/ (stored 0%)
  adding: content/posts/images/spdde/CzLGA9jtJuj.jpg (deflated 2%)
  adding: content/posts/images/stateofisrael/ (stored 0%)
  adding: content/posts/images/stateofisrael/CzGoQoFrGQR.jpg (deflated 1%)
  adding: content/posts/videos/ (stored 0%)
  adding: content/posts/videos/stateofisrael/ (stored 0%)
  adding: content/posts/videos/stateofisrael/CzLuRzfrukl.mp4 (deflated 0%)


The file `posts.zip` will appear in the files pane on the left. Right click the file to download it. Files can also be moved to Google Drive (faster!): `!cp posts.zip /content/drive/MyDrive/`.